# Advanced Analytics

This notebook computes fund risk, investor behavior, portfolio concentration, and recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
OUT = Path("output")
fm = pd.read_csv("01_fund_master.csv")
nav = pd.read_csv("02_nav_history.csv")
perf = pd.read_csv("07_scheme_performance.csv")
txn = pd.read_csv("08_investor_transactions.csv")
hold = pd.read_csv("09_portfolio_holdings.csv")
bench = pd.read_csv("10_benchmark_indices.csv")
nav["date"] = pd.to_datetime(nav["date"])
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"])
bench["date"] = pd.to_datetime(bench["date"])


## 1. Historical VaR and CVaR

VaR at 95% is the 5th percentile of daily returns. CVaR is the mean return below that threshold.

In [ ]:
navr = nav.sort_values(["amfi_code","date"]).copy()
navr["daily_return"] = navr.groupby("amfi_code")["nav"].pct_change()
rows=[]
for code, g in navr.groupby("amfi_code"):
    r = g["daily_return"].dropna()
    if len(r) < 5:
        continue
    var95 = np.nanpercentile(r, 5)
    cvar = r[r <= var95].mean()
    rows.append({"amfi_code": code, "VaR_95": var95, "CVaR": cvar, "n_obs": len(r)})
var_cvar = pd.DataFrame(rows).merge(fm[["amfi_code","scheme_name","fund_house","category","plan"]], on="amfi_code", how="left")
var_cvar.to_csv(OUT/"var_cvar_report.csv", index=False)
var_cvar.head()

## 2. Rolling 90-day Sharpe

Rolling Sharpe uses 90-day mean returns divided by 90-day volatility, annualized by sqrt(252).

In [ ]:
key_codes = navr["amfi_code"].drop_duplicates().astype(str).head(5).tolist()
plt.figure(figsize=(12,6))
for code in key_codes:
    g = navr[navr["amfi_code"].astype(str)==code].set_index("date")["daily_return"]
    sharpe = g.rolling(90).mean() / g.rolling(90).std() * np.sqrt(252)
    plt.plot(sharpe.index, sharpe.values, label=code, linewidth=1.5)
plt.legend(title="amfi_code", ncol=2, fontsize=8)
plt.title("Rolling 90-day Sharpe (5 funds)")
plt.tight_layout()
plt.savefig(OUT/"rolling_sharpe_chart.png", dpi=200)
plt.close()
key_codes

## 3. Investor Cohort Analysis

In [ ]:
txn = txn.sort_values(["investor_id","transaction_date"]).copy()
first_year = txn.groupby("investor_id")["transaction_date"].min().dt.year.rename("cohort_year")
txn = txn.merge(first_year, on="investor_id", how="left")
cohort = txn.groupby(["cohort_year","investor_id"]).agg(avg_sip_amount=("amount_inr", lambda s: s[txn.loc[s.index,"transaction_type"].eq("SIP")].mean()), total_invested=("amount_inr","sum")).reset_index()
cohort_summary = cohort.groupby("cohort_year").agg(avg_sip_amount=("avg_sip_amount","mean"), total_invested=("total_invested","sum"), investors=("investor_id","nunique")).reset_index()
prefer = txn.groupby(["cohort_year","amfi_code"]).size().reset_index(name="count").sort_values(["cohort_year","count"], ascending=[True,False])
prefer = prefer.drop_duplicates("cohort_year").merge(fm[["amfi_code","scheme_name"]], on="amfi_code", how="left")
cohort_summary.merge(prefer[["cohort_year","scheme_name"]], on="cohort_year", how="left").head()

## 4. SIP Continuity Analysis

In [ ]:
sip = txn[txn["transaction_type"].eq("SIP")].sort_values(["investor_id","transaction_date"]).copy()
sip["gap_days"] = sip.groupby("investor_id")["transaction_date"].diff().dt.days
cont = sip.groupby("investor_id").agg(sip_count=("transaction_date","count"), avg_gap_days=("gap_days","mean"), max_gap_days=("gap_days","max")).reset_index()
cont = cont[cont["sip_count"] >= 6].copy()
cont["at_risk"] = cont["max_gap_days"].fillna(0) > 35
cont.head()

## 5. Simple Fund Recommender

In [ ]:
def recommend(risk_appetite):
    risk_map = {"Low": ["Low"], "Moderate": ["Moderate","Medium"], "High": ["High"]}
    vals = risk_map.get(str(risk_appetite).title(), [risk_appetite])
    df = perf.copy()
    if "risk_grade" not in df.columns:
        return pd.DataFrame(columns=["scheme_name","fund_house","risk_grade","sharpe_ratio"])
    sub = df[df["risk_grade"].astype(str).str.title().isin(vals)].copy()
    return sub.sort_values("sharpe_ratio", ascending=False).head(3)[["scheme_name","fund_house","risk_grade","sharpe_ratio"]]

recommend("Moderate")

## 6. Sector HHI Concentration

In [ ]:
if "sector" in hold.columns and "weight_pct" in hold.columns:
    hhi = hold[hold["category"].astype(str).str.contains("Equity", na=False)].copy()
    hhi = hhi.groupby(["amfi_code","sector"], dropna=False)["weight_pct"].sum().reset_index()
    hhi["w"] = hhi["weight_pct"] / 100.0
    hhi = hhi.groupby("amfi_code").apply(lambda x: pd.Series({"HHI": (x["w"]**2).sum()})).reset_index()
    hhi = hhi.merge(fm[["amfi_code","scheme_name","category"]], on="amfi_code", how="left")
    hhi.head()
else:
    pd.DataFrame({"note":["portfolio_holdings file does not expose sector and weight_pct columns in this schema"]})

## 7. Advanced Insights

- Funds with the most negative VaR and deepest CVaR are the riskiest on daily downside.
- Cohorts with the largest total invested size are the dominant capital source.
- A high share of SIP investors with long gaps indicates weaker continuity.
- The recommender highlights the highest Sharpe funds inside each risk bucket.
- Concentrated portfolios with high HHI deserve caution even when returns look strong.